# 01 - EDA e Baselines

Este notebook realiza uma analise exploratoria inicial do dataset de churn e compara baselines simples com Scikit-Learn. Ele usa o mesmo pacote `churn` do projeto para evitar divergencia entre notebook e codigo de producao.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from churn.config import RANDOM_SEED  # noqa: E402
from churn.data import load_churn_csv, split_features_target  # noqa: E402
from churn.features import build_preprocessor  # noqa: E402
from churn.metrics import binary_classification_metrics  # noqa: E402

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "churn.csv"

## 1. Leitura dos dados

In [ ]:
df = load_churn_csv(DATA_PATH)
df.shape

In [ ]:
df.head()

## 2. Schema, nulos e distribuicao do alvo

In [ ]:
df.dtypes

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
target_distribution = df["Churn"].value_counts(normalize=True).rename("proportion")
target_distribution

In [ ]:
counts = df["Churn"].value_counts().sort_index()
ax = counts.plot(kind="bar", color=["#4C78A8", "#F58518"])
ax.set_title("Distribuicao da Classe Churn")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade")
ax.set_xticklabels(["Nao churn", "Churn"], rotation=0)
plt.tight_layout()

## 3. Baselines Scikit-Learn

Avaliamos um baseline ingenuo (`DummyClassifier`) e uma regressao logistica balanceada. A separacao final usa 80% treino e 20% teste, com estratificacao do alvo. A validacao cruzada roda apenas nos 80% de treino.

In [ ]:
x, y = split_features_target(df)
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_SEED,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
}

rows = []
for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(x_train)),
            ("classifier", estimator),
        ]
    )
    cv_results = cross_validate(pipeline, x_train, y_train, cv=cv, scoring=scoring)
    pipeline.fit(x_train, y_train)

    if hasattr(pipeline.named_steps["classifier"], "predict_proba"):
        y_prob = pipeline.predict_proba(x_test)[:, 1]
    else:
        y_prob = pipeline.predict(x_test)

    test_metrics = binary_classification_metrics(y_test, y_prob)
    row = {"model": model_name}
    row.update(
        {
            f"cv_{metric}": float(np.mean(cv_results[f"test_{metric}"]))
            for metric in scoring
        }
    )
    row.update({f"test_{metric}": value for metric, value in test_metrics.items()})
    rows.append(row)

results = pd.DataFrame(rows).sort_values("test_f1", ascending=False)
results

## 4. Observacoes

- Accuracy pode ser enganosa quando a classe churn e minoritaria.
- Recall, F1 e ROC-AUC sao metricas mais importantes para comparar modelos de churn.
- O baseline logistico balanceado serve como referencia minima para a MLP PyTorch.